In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
df = pd.read_csv('apartments.csv').drop(22)

In [9]:
df.head(2)

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...","['Swimming Pool', 'Salon', 'Restaurant', 'Spa'..."
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...","['Bowling Alley', 'Mini Theatre', 'Manicured G..."


In [50]:
df.iloc[0]['TopFacilities']

'Swimming Pool Salon Restaurant Spa Cafeteria Sun Deck 24x7 Security Club House Gated Community'

In [51]:
df.iloc[0]['PriceDetails']

"{'2 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,370 sq.ft.', 'price-range': '₹ 2 - 2.4 Cr'}, '3 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,850 - 2,050 sq.ft.', 'price-range': '₹ 2.25 - 3.59 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '2,600 sq.ft.', 'price-range': '₹ 3.24 - 4.56 Cr'}}"

In [52]:
df.iloc[0]['LocationAdvantages']

"{'Bajghera Road': '800 Meter', 'Palam Vihar Halt': '2.5 KM', 'DPSG Palam Vihar': '3.1 KM', 'Park Hospital': '3.1 KM', 'Gurgaon Railway Station': '4.9 KM', 'The NorthCap University': '5.4 KM', 'Dwarka Expy': '1.2 KM', 'Hyatt Place Gurgaon Udyog Vihar': '7.7 KM', 'Dwarka Sector 21, Metro Station': '7.2 KM', 'Pacific D21 Mall': '7.4 KM', 'Indira Gandhi International Airport': '14.7 KM', 'Hamoni Golf Camp': '6.2 KM', 'Fun N Food Waterpark': '8.8 KM', 'Accenture DDC5': '9 KM'}"

In [53]:
import ast

def list_string_to_text(data):
    if not isinstance(data, str):
        return ""

    try:
        arr = ast.literal_eval(data)  # Convert string to list safely
        if isinstance(arr, list):
            return " ".join(map(str, arr))
    except (ValueError, SyntaxError):
        return ""

    return ""

In [54]:
df['TopFacilities']=df['TopFacilities'].apply(list_string_to_text)

In [55]:
df.head(2)

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...",
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...",


In [56]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

In [57]:
tfidf_matrix=tfidf_vectorizer.fit_transform(df['TopFacilities'])

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [ ]:
tfidf_matrix.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.14384358,
        0.14384358],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(246, 973))

In [ ]:
cosine_sim1=cosine_similarity(tfidf_matrix,tfidf_matrix)

In [ ]:
cosine_sim1

array([[1.        , 0.01095159, 0.        , ..., 0.01183329, 0.08656385,
        0.0110727 ],
       [0.01095159, 1.        , 0.01982808, ..., 0.11904241, 0.01555534,
        0.00963852],
       [0.        , 0.01982808, 1.        , ..., 0.07022934, 0.03821637,
        0.01963506],
       ...,
       [0.01183329, 0.11904241, 0.07022934, ..., 1.        , 0.09825738,
        0.03255851],
       [0.08656385, 0.01555534, 0.03821637, ..., 0.09825738, 1.        ,
        0.06257614],
       [0.0110727 , 0.00963852, 0.01963506, ..., 0.03255851, 0.06257614,
        1.        ]], shape=(246, 246))

In [ ]:
def recommend_properties(property_name, cosine_sim=cosine_sim1):
    # Get the index of the property that matches the name
    idx = df.index[df['PropertyName'] == property_name].tolist()[0]

    # Get the pairwise similarity scores with that property
    sim_scores = list(enumerate(cosine_sim1[idx]))

    # Sort the properties based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 5 most similar properties
    sim_scores = sim_scores[1:6]

    # Get the property indices
    property_indices = [i[0] for i in sim_scores]
    
    recommendations_df = pd.DataFrame({
        'PropertyName': df['PropertyName'].iloc[property_indices],
        'SimilarityScore': sim_scores
    })

    # Return the top 5 most similar properties
    return recommendations_df

In [ ]:
recommend_properties("DLF The Arbour")

,PropertyName,SimilarityScore
64,Ace Palm Floors,"(63, 0.551340635885514)"
93,JMS The Nation,"(92, 0.48399273152019534)"
217,Yashika 104,"(216, 0.4791925565137415)"
18,Whiteland Blissville,"(18, 0.46907955257042044)"
63,Vatika Aspiration,"(62, 0.4160816171209393)"


In [ ]:
# PriceDetails

In [ ]:
df['PriceDetails'][0]

"{'2 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,370 sq.ft.', 'price-range': '₹ 2 - 2.4 Cr'}, '3 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '1,850 - 2,050 sq.ft.', 'price-range': '₹ 2.25 - 3.59 Cr'}, '4 BHK': {'building_type': 'Apartment', 'area_type': 'Carpet Area', 'area': '2,600 sq.ft.', 'price-range': '₹ 3.24 - 4.56 Cr'}}"

In [ ]:
import pandas as pd
import json



# Function to parse and extract the required features from the PriceDetails column
def refined_parse_modified_v2(detail_str):
    try:
        details = json.loads(detail_str.replace("'", "\""))
    except:
        return {}

    extracted = {}
    for bhk, detail in details.items():
        # Extract building type
        extracted[f'building type_{bhk}'] = detail.get('building_type')

        # Parsing area details
        area = detail.get('area', '')
        area_parts = area.split('-')
        if len(area_parts) == 1:
            try:
                value = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area low {bhk}'] = value
                extracted[f'area high {bhk}'] = value
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None
        elif len(area_parts) == 2:
            try:
                extracted[f'area low {bhk}'] = float(area_parts[0].replace(',', '').replace(' sq.ft.', '').strip())
                extracted[f'area high {bhk}'] = float(area_parts[1].replace(',', '').replace(' sq.ft.', '').strip())
            except:
                extracted[f'area low {bhk}'] = None
                extracted[f'area high {bhk}'] = None

        # Parsing price details
        price_range = detail.get('price-range', '')
        price_parts = price_range.split('-')
        if len(price_parts) == 2:
            try:
                extracted[f'price low {bhk}'] = float(price_parts[0].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                extracted[f'price high {bhk}'] = float(price_parts[1].replace('₹', '').replace(' Cr', '').replace(' L', '').strip())
                if 'L' in price_parts[0]:
                    extracted[f'price low {bhk}'] /= 100
                if 'L' in price_parts[1]:
                    extracted[f'price high {bhk}'] /= 100
            except:
                extracted[f'price low {bhk}'] = None
                extracted[f'price high {bhk}'] = None

    return extracted
# Apply the refined parsing and generate the new DataFrame structure
data_refined = []

for _, row in df.iterrows():
    features = refined_parse_modified_v2(row['PriceDetails'])
    
    # Construct a new row for the transformed dataframe
    new_row = {'PropertyName': row['PropertyName']}
    
    # Populate the new row with extracted features
    for config in ['1 BHK', '2 BHK', '3 BHK', '4 BHK', '5 BHK', '6 BHK', '1 RK', 'Land']:
        new_row[f'building type_{config}'] = features.get(f'building type_{config}')
        new_row[f'area low {config}'] = features.get(f'area low {config}')
        new_row[f'area high {config}'] = features.get(f'area high {config}')
        new_row[f'price low {config}'] = features.get(f'price low {config}')
        new_row[f'price high {config}'] = features.get(f'price high {config}')
    
    data_refined.append(new_row)

df_final_refined_v2 = pd.DataFrame(data_refined).set_index('PropertyName')

In [ ]:
df_final_refined_v2['building type_Land'] = df_final_refined_v2['building type_Land'].replace({'':'Land'})

In [ ]:
df_final_refined_v2

,building type_1 BHK,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,building type_2 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,...,building type_1 RK,area low 1 RK,area high 1 RK,price low 1 RK,price high 1 RK,building type_Land,area low Land,area high Land,price low Land,price high Land
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,NaN,NaN,NaN,NaN,NaN,Apartment,1370.0,1370.0,2.0000,2.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
M3M Crown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Adani Brahma Samsara Vilasa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Land,500.0,4329.0,2.05,41.13
Sobha City,NaN,NaN,NaN,NaN,NaN,Apartment,1381.0,1692.0,1.5500,3.21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Signature Global City 93,NaN,NaN,NaN,NaN,NaN,Independent Floor,981.0,1118.0,0.9301,1.06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,NaN,NaN,NaN,NaN,NaN,Apartment,964.0,964.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Pyramid Urban Homes 2,Apartment,335.0,398.0,23.45,0.2786,Apartment,500.0,625.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Satya The Hermitage,NaN,NaN,NaN,NaN,NaN,Apartment,1450.0,1450.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final_refined_v2.info()

<class 'pandas.DataFrame'>
Index: 246 entries, Smartworld One DXP to SS The Coralwood
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   building type_1 BHK  18 non-null     str    
 1   area low 1 BHK       18 non-null     float64
 2   area high 1 BHK      18 non-null     float64
 3   price low 1 BHK      5 non-null      float64
 4   price high 1 BHK     5 non-null      float64
 5   building type_2 BHK  121 non-null    str    
 6   area low 2 BHK       121 non-null    float64
 7   area high 2 BHK      121 non-null    float64
 8   price low 2 BHK      72 non-null     float64
 9   price high 2 BHK     72 non-null     float64
 10  building type_3 BHK  186 non-null    str    
 11  area low 3 BHK       186 non-null    float64
 12  area high 3 BHK      186 non-null    float64
 13  price low 3 BHK      133 non-null    float64
 14  price high 3 BHK     133 non-null    float64
 15  building type_4 BHK  144 n

In [ ]:
categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_20448\1646369994.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_final_refined_v2.select_dtypes(include=['object']).columns.tolist()


In [ ]:
categorical_columns

['building type_1 BHK',
 'building type_2 BHK',
 'building type_3 BHK',
 'building type_4 BHK',
 'building type_5 BHK',
 'building type_6 BHK',
 'building type_1 RK',
 'building type_Land']

In [ ]:
ohe_df=pd.get_dummies(df_final_refined_v2,columns=categorical_columns,drop_first=True,dtype=int)

In [ ]:
ohe_df.fillna(0,inplace=True)

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,...,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,0.0,0.0,0.00,0.0000,1370.0,1370.0,2.0000,2.40,1850.0,2050.0,...,0,0,0,0,0,0,0,0,0,0
M3M Crown,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1605.0,2170.0,...,0,0,0,0,0,0,0,0,0,0
Adani Brahma Samsara Vilasa,0.0,0.0,0.00,0.0000,0.0,0.0,0.0000,0.00,1800.0,3150.0,...,0,0,1,0,0,1,0,0,0,0
Sobha City,0.0,0.0,0.00,0.0000,1381.0,1692.0,1.5500,3.21,1711.0,2343.0,...,0,0,0,0,0,0,0,0,0,0
Signature Global City 93,0.0,0.0,0.00,0.0000,981.0,1118.0,0.9301,1.06,1235.0,1530.0,...,1,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DLF Princeton Estate,0.0,0.0,0.00,0.0000,964.0,964.0,0.0000,0.00,1127.0,1127.0,...,0,0,0,0,0,0,0,0,0,0
Pyramid Urban Homes 2,335.0,398.0,23.45,0.2786,500.0,625.0,0.0000,0.00,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
Satya The Hermitage,0.0,0.0,0.00,0.0000,1450.0,1450.0,0.0000,0.00,1991.0,1991.0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
ohe_df.head(2)

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,...,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,0.0,0.0,0.0,0.0,1370.0,1370.0,2.0,2.4,1850.0,2050.0,...,0,0,0,0,0,0,0,0,0,0
M3M Crown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1605.0,2170.0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
ohe_df_normalized=pd.DataFrame(scaler.fit_transform(ohe_df),columns=ohe_df.columns,index=ohe_df.index)

In [ ]:
ohe_df_normalized.head()

,area low 1 BHK,area high 1 BHK,price low 1 BHK,price high 1 BHK,area low 2 BHK,area high 2 BHK,price low 2 BHK,price high 2 BHK,area low 3 BHK,area high 3 BHK,...,building type_2 BHK_Independent Floor,building type_2 BHK_Service Apartment,building type_3 BHK_Independent Floor,building type_3 BHK_Service Apartment,building type_3 BHK_Villa,building type_4 BHK_Independent Floor,building type_4 BHK_Villa,building type_5 BHK_Independent Floor,building type_5 BHK_Villa,building type_6 BHK_Villa
PropertyName,,,,,,,,,,,,,,,,,,,,,
Smartworld One DXP,-0.252266,-0.169584,-0.105197,-0.082332,1.223499,1.020101,-0.173712,1.158423,0.553787,0.370864,...,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
M3M Crown,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.293086,0.472749,...,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Adani Brahma Samsara Vilasa,-0.252266,-0.169584,-0.105197,-0.082332,-0.893541,-0.896660,-0.283546,-0.387986,0.500583,1.304803,...,-0.289310,-0.063888,2.683282,-0.063888,-0.171139,3.924283,-0.236208,-0.111111,-0.216353,-0.063888
Sobha City,-0.252266,-0.169584,-0.105197,-0.082332,1.240497,1.470610,-0.198425,1.680336,0.405879,0.619632,...,-0.289310,-0.063888,-0.372678,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888
Signature Global City 93,-0.252266,-0.169584,-0.105197,-0.082332,0.622383,0.667529,-0.232468,0.295011,-0.100626,-0.070634,...,3.456497,-0.063888,2.683282,-0.063888,-0.171139,-0.254824,-0.236208,-0.111111,-0.216353,-0.063888


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix
cosine_sim2 = cosine_similarity(ohe_df_normalized)

In [58]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim2[ohe_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = ohe_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    
    return recommendations_df

recommend_properties_with_scores('M3M Golf Hills')

,PropertyName,SimilarityScore
0,AIPL The Peaceful Homes,0.955462
1,Smartworld One DXP,0.954670
2,Unitech Escape,0.953092
3,M3M Capital,0.951156
4,BPTP Terra,0.943128
...,...,...
240,Golden Park,-0.522391
241,Satya Merano Greens,-0.523660
242,ROF Normanton Park,-0.525129
243,BPTP Green Oaks,-0.525286


In [59]:
def distance_to_meters(distance_str):
    try:
        if 'Km' in distance_str or 'KM' in distance_str:
            return float(distance_str.split()[0]) * 1000
        elif 'Meter' in distance_str or 'meter' in distance_str:
            return float(distance_str.split()[0])
        else:
            return None
    except:
        return None


In [60]:
# Extract distances for each location
location_matrix = {}
for index, row in df.iterrows():
    distances = {}
    for location, distance in ast.literal_eval(row['LocationAdvantages']).items():
        distances[location] = distance_to_meters(distance)
    location_matrix[index] = distances

# Convert the dictionary to a dataframe
location_df = pd.DataFrame.from_dict(location_matrix, orient='index')

# Display the first few rows
location_df.head()

,Bajghera Road,Palam Vihar Halt,DPSG Palam Vihar,Park Hospital,Gurgaon Railway Station,The NorthCap University,Dwarka Expy,Hyatt Place Gurgaon Udyog Vihar,"Dwarka Sector 21, Metro Station",Pacific D21 Mall,...,MCC Cricket Ground Dhankot,The Shri Ram School Aravali,Taj City Centre Gurugram,Minda Industries Corporate Office,"Rampura Flyover, Naurangpur Rd",Manesar toll plaza - Kherki Daula,"Imt Manesar, Gurugram",Holiday Inn,Sector 84 Road,Skyview Corporate Park
0,800.0,2500.0,3100.0,3100.0,4900.0,5400.0,1200.0,7700.0,7200.0,7400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,550.0,NaN,NaN,NaN,NaN,6700.0,3800.0,NaN,NaN,7500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37,5300.0,NaN,NaN,NaN,2500.0,8800.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,1500.0,NaN,NaN,NaN,6500.0,6700.0,5100.0,NaN,NaN,8200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,5500.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
location_df.index = df.PropertyName

In [100]:
location_df.columns[10:100]

Index(['Aapno Ghar', 'AapnoGhar', 'Aarvy Healthcare',
       'Aarvy Healthcare Hospital', 'Aarvy Healthcare Super Speciality',
       'Aarvy Hospital', 'Aatish Hospital', 'Accenture DDC5',
       'Adarsh Senior Secondary School', 'Adarsh public school,Garhi Harsaru',
       'Agri Business Management Collage', 'Airia Mall',
       'Airia Mall Sector 68', 'Airport', 'Ajit Stadium Dhanwapur',
       'Alfaa Health Care Hospital', 'Alpine Convent School',
       'Alpine Hospital', 'Alpine School', 'Altrade Business Centre',
       'Aman Hospital', 'Aman Hospital & Surgical Centre', 'Ambience Mall',
       'Ambience Mall New', 'Ambience Public School', 'American Express',
       'Amity', 'Amity University', 'Amity University Gurugram',
       'Amma Hospital', 'Anand Multispeciality Hospital', 'Anand Preschool',
       'Ananta Hospital', 'Ansal Plaza', 'Anya Gurgaon',
       'Ap Sports cricket ground', 'Apex Plus Hospital', 'Apollo',
       'Apollo Pharmacy', 'Approved Sector 37 Mero Station'

In [65]:
!pip install rapidfuzz

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   -------------------------- ------------- 1.0/1.6 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 7.6 MB/s  0:00:00



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [101]:
import re

def normalize_location(name):
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    # Standardize common abbreviations
    replacements = {
        r"\bintl\b": "international",
        r"\bint\.\b": "international",
        r"\brd\b": "road",
        r"\bexpy\b": "expressway",
        r"\bhwy\b": "highway",
        r"\bstn\b": "station",
        r"\bmetro stn\b": "metro station",
        r"\bhosp\b": "hospital",
        r"\buniv\b": "university",
        r"\bsec\b": "sector",
        r"\bgurgaon\b": "gurugram",
        r"&": "and"
    }

    for pattern, replacement in replacements.items():
        name = re.sub(pattern, replacement, name)

    # Remove punctuation
    name = re.sub(r"[-_/(),.]", " ", name)

    # Remove duplicate spaces
    name = re.sub(r"\s+", " ", name)

    return name.strip()

In [102]:
from rapidfuzz import fuzz

columns = list(location_df.columns)

normalized = {col: normalize_location(col) for col in columns}

mapping = {}
visited = set()

THRESHOLD = 95

for col1 in columns:

    if col1 in visited:
        continue

    mapping[col1] = col1
    visited.add(col1)

    for col2 in columns:

        if col2 == col1 or col2 in visited:
            continue

        score = fuzz.token_sort_ratio(
            normalized[col1],
            normalized[col2]
        )

        if score >= THRESHOLD:
            mapping[col2] = col1
            visited.add(col2)

In [103]:
for old, new in mapping.items():
    if old != new:
        print(f"{old:45} ---> {new}")

Euro International School, Sector- 51         ---> Euro International School, Sector- 109


In [104]:
mapping["Euro Intl School Sec-51"] = "Euro International School, Sector- 51"
mapping["Euro International School, sector- 51"] = "Euro International School, Sector- 51"
mapping["NH 48"] = "National Highway  48"

mapping["NH -8"] = "National Highway  48"

In [106]:
from rapidfuzz import fuzz
import pandas as pd

pairs = []

for i, c1 in enumerate(columns):
    for c2 in columns[i + 1:]:

        score = fuzz.token_sort_ratio(
            normalize_location(c1),
            normalize_location(c2)
        )

        if score >= 90:
            pairs.append([c1, c2, score])

review_df = pd.DataFrame(
    pairs,
    columns=["Location 1", "Location 2", "Similarity"]
).sort_values("Similarity", ascending=False)

review_df.head(50)

,Location 1,Location 2,Similarity
14,"Euro International School, Sector- 109","Euro International School, Sector- 51",95.774648
0,Aapno Ghar,AapnoGhar,94.736842
40,Signature Super Speciality Hospital,The Signature Super Speciality Hospital,94.594595
11,"Euro International School, Sec 84","Euro International School, Sector- 51",94.285714
36,Sector 53-54 Metro Station,Sector 55 Metro Station,93.877551
38,Sector 55 Metro Station,Sector 55-56 Metro Station,93.877551
34,Sapphire 83 Mall,Sapphire 93 Mall,93.750000
2,Artemis Hospital,Artimis hospital,93.750000
19,Huda City Centre,Huda city center,93.750000
41,Southern Peripheral Rd,Southern Periphery Road,93.617021


In [107]:
location_df = location_df.rename(columns=mapping)

In [108]:
location_df = location_df.T.groupby(level=0).min().T

In [109]:
location_df.shape

(246, 957)

In [110]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [111]:
location_df.shape

(246, 957)

In [114]:
# location_df.fillna(54000,inplace=True)

In [115]:
location_df.head()

,AIIMS,AIIMS Jhajjar,AIPL Business Centre,AIPL Business Club,AIPL Business Club Sector 62,AIPL Business Co Working Space,AIPL Business Tower,AIPL Joy Street Mall,APJ Abdul Kalam Park,ASF Insignia SEZ,Aapno Ghar,AapnoGhar,Aarvy Healthcare,Aarvy Healthcare Hospital,Aarvy Healthcare Super Speciality,Aarvy Hospital,Aatish Hospital,Accenture DDC5,Adarsh Senior Secondary School,"Adarsh public school,Garhi Harsaru",Agri Business Management Collage,Airia Mall,Airia Mall Sector 68,Airport,Ajit Stadium Dhanwapur,Alfaa Health Care Hospital,Alpine Convent School,Alpine Hospital,Alpine School,Altrade Business Centre,Aman Hospital,Aman Hospital & Surgical Centre,Ambience Mall,Ambience Mall New,Ambience Public School,American Express,Amity,Amity University,Amity University Gurugram,Amma Hospital,Anand Multispeciality Hospital,Anand Preschool,Ananta Hospital,Ansal Plaza,Anya Gurgaon,Ap Sports cricket ground,Apex Plus Hospital,Apollo,Apollo Pharmacy,Approved Sector 37 Mero Station,Appu Ghar,Appu Ghar Water Park,Aradhya Cricket Club Gurgaon,Aravali Adventure Hill,Aravalli Hill View Point,Aravalli Hills,Arc Hospital,Arc Multi Speciality,Arc Multi Speciality Hospital,Ardee Mall,Artemis,Artemis Hospital,Artemis Hospital Gurgaon,Artimis hospital,Aryan Hospital,Ascendas OneHub Gurgaon,Ascendas OneHub Gurgaon Business Park,Ashiana Anmol Kid Centric Homes,Ashoka International School,Athena,Au Grand Air,Axis Bank,Axis Bank ATM,"Axis Bank, Sohna Rd",BM College of Technology,BM College of Technology & Mgmt,BML Munjal University (BMU),BOB ATM,BSF Golf Course,Baba Kanala Chowk,Badsa AMS Hospital,Badshahpur Sohna Hwy,Badshahpur Sohna Rd Hwy,"Badshahpur Sohna Rd Hwy, Haryana","Badshahpur Sohna Rd Hwy, Malibu Town","Badshahpur Sohna Rd Hwy, Raghav Vatika","Badshahpur Sohna Rd Hwy, Rajoria Ngr","Badshahpur Sohna Rd Hwy,Sector 48",Baghera University,Bajghera Road,Bal Bharati Public School,Balaji Hospital,Bamroli Cricket Ground,Banjara Market Gurugram,Bank Of Baroda,Bank Of Baroda ATM,Basai Dhancourt Railway Station,Basai Dhankot,Basai Dhankot Railway Station,Basai Enclave Park,Basai Metro Station,Basai Road,Basant Valley Global School,Best IVF Centre,Bestech Business Tower,Bestech Central Square Mall,Bharat Petrol Pump,Bharat Petroleum Petrol Pump,Bharat Petroleum Retail Outlet,Bharat Petroleum Shree Shyam Filling,Bharat Ram Global School,Bharat Singh fuel company,Bharti International Convent School,Bhondsi Nature Park,BigBazaar,Bijwasan Railway Station,Biryani Shah,Blue Bells Public School,Broadways International School,CBR Cricket Ground,CD International School,CK Birla Hospital,"CK Birla Hospital, Gurgaon",CNG Petrol Pump,Cambridge College Of Education,Cambridge Montessori,Cambridge Montessori Preschool,Cambridge Pre-School,Canara Bank,Canara Bank - Nawada Fatehpur,Canara Bank ATM,Canara Bank New Palam Vihar,"Candor TechSpace, Sector 48",Candor Techspace,Capital Business Park,Capital Cyberscape,Captain Chandan Lal Marg,Central Bank Of India Sohna Rd,Central Park Flower Valley,Central Park II Road,Central Park Resorts,"Central Park, Sohna Rd",Central Peripheral Road,Central Plaza Mall,Centrum Plaza,Chauma Road,Cherub's Cradle,Children Park,Chirag Hospital,Choice pharmacy,Citibank ATM,City Hospital,City Square,Civil Hospital,Cloudnine Hospital Sector 47,CoNexus.Life B35,Colonel's Central Academy,Conscient One,Conscient One Mall,Cool Deck Coffee,Country Inn,Country Inn & Suites by Radisson,Creative Tennis Academy,Cricket Academy,Cyber ​​Park,DLC Cricket Ground,DLF Corporate Greens,DLF Corporate Park,DLF Cyber City,DLF Golf and Country Club,DLF Grand Mall,DLF Linear Park,DLF Site central office,DLF World Tech Park,DLF5 Summit Plaza,DPG Degree College,DPG Institute of Technology,DPGITM Engineering College Sector 34,DPS,DPS International Edge,DPS International School,DPS Manesar,DPS Sector 103,DPSG Palam Vihar,DPSG Palam Vihar Gurugram,DSD College,Damdama Lake,Damdama Lake Rd,Damdama More,Daultabad Stadium,Daultabad Village Park,De Adventure Amusement Park,De Adve

In [116]:
from sklearn.preprocessing import StandardScaler
# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler to the entire dataframe
location_df_normalized = pd.DataFrame(scaler.fit_transform(location_df), columns=location_df.columns, index=location_df.index)

In [117]:
location_df_normalized.head()

,AIIMS,AIIMS Jhajjar,AIPL Business Centre,AIPL Business Club,AIPL Business Club Sector 62,AIPL Business Co Working Space,AIPL Business Tower,AIPL Joy Street Mall,APJ Abdul Kalam Park,ASF Insignia SEZ,Aapno Ghar,AapnoGhar,Aarvy Healthcare,Aarvy Healthcare Hospital,Aarvy Healthcare Super Speciality,Aarvy Hospital,Aatish Hospital,Accenture DDC5,Adarsh Senior Secondary School,"Adarsh public school,Garhi Harsaru",Agri Business Management Collage,Airia Mall,Airia Mall Sector 68,Airport,Ajit Stadium Dhanwapur,Alfaa Health Care Hospital,Alpine Convent School,Alpine Hospital,Alpine School,Altrade Business Centre,Aman Hospital,Aman Hospital & Surgical Centre,Ambience Mall,Ambience Mall New,Ambience Public School,American Express,Amity,Amity University,Amity University Gurugram,Amma Hospital,Anand Multispeciality Hospital,Anand Preschool,Ananta Hospital,Ansal Plaza,Anya Gurgaon,Ap Sports cricket ground,Apex Plus Hospital,Apollo,Apollo Pharmacy,Approved Sector 37 Mero Station,Appu Ghar,Appu Ghar Water Park,Aradhya Cricket Club Gurgaon,Aravali Adventure Hill,Aravalli Hill View Point,Aravalli Hills,Arc Hospital,Arc Multi Speciality,Arc Multi Speciality Hospital,Ardee Mall,Artemis,Artemis Hospital,Artemis Hospital Gurgaon,Artimis hospital,Aryan Hospital,Ascendas OneHub Gurgaon,Ascendas OneHub Gurgaon Business Park,Ashiana Anmol Kid Centric Homes,Ashoka International School,Athena,Au Grand Air,Axis Bank,Axis Bank ATM,"Axis Bank, Sohna Rd",BM College of Technology,BM College of Technology & Mgmt,BML Munjal University (BMU),BOB ATM,BSF Golf Course,Baba Kanala Chowk,Badsa AMS Hospital,Badshahpur Sohna Hwy,Badshahpur Sohna Rd Hwy,"Badshahpur Sohna Rd Hwy, Haryana","Badshahpur Sohna Rd Hwy, Malibu Town","Badshahpur Sohna Rd Hwy, Raghav Vatika","Badshahpur Sohna Rd Hwy, Rajoria Ngr","Badshahpur Sohna Rd Hwy,Sector 48",Baghera University,Bajghera Road,Bal Bharati Public School,Balaji Hospital,Bamroli Cricket Ground,Banjara Market Gurugram,Bank Of Baroda,Bank Of Baroda ATM,Basai Dhancourt Railway Station,Basai Dhankot,Basai Dhankot Railway Station,Basai Enclave Park,Basai Metro Station,Basai Road,Basant Valley Global School,Best IVF Centre,Bestech Business Tower,Bestech Central Square Mall,Bharat Petrol Pump,Bharat Petroleum Petrol Pump,Bharat Petroleum Retail Outlet,Bharat Petroleum Shree Shyam Filling,Bharat Ram Global School,Bharat Singh fuel company,Bharti International Convent School,Bhondsi Nature Park,BigBazaar,Bijwasan Railway Station,Biryani Shah,Blue Bells Public School,Broadways International School,CBR Cricket Ground,CD International School,CK Birla Hospital,"CK Birla Hospital, Gurgaon",CNG Petrol Pump,Cambridge College Of Education,Cambridge Montessori,Cambridge Montessori Preschool,Cambridge Pre-School,Canara Bank,Canara Bank - Nawada Fatehpur,Canara Bank ATM,Canara Bank New Palam Vihar,"Candor TechSpace, Sector 48",Candor Techspace,Capital Business Park,Capital Cyberscape,Captain Chandan Lal Marg,Central Bank Of India Sohna Rd,Central Park Flower Valley,Central Park II Road,Central Park Resorts,"Central Park, Sohna Rd",Central Peripheral Road,Central Plaza Mall,Centrum Plaza,Chauma Road,Cherub's Cradle,Children Park,Chirag Hospital,Choice pharmacy,Citibank ATM,City Hospital,City Square,Civil Hospital,Cloudnine Hospital Sector 47,CoNexus.Life B35,Colonel's Central Academy,Conscient One,Conscient One Mall,Cool Deck Coffee,Country Inn,Country Inn & Suites by Radisson,Creative Tennis Academy,Cricket Academy,Cyber ​​Park,DLC Cricket Ground,DLF Corporate Greens,DLF Corporate Park,DLF Cyber City,DLF Golf and Country Club,DLF Grand Mall,DLF Linear Park,DLF Site central office,DLF World Tech Park,DLF5 Summit Plaza,DPG Degree College,DPG Institute of Technology,DPGITM Engineering College Sector 34,DPS,DPS International Edge,DPS International School,DPS Manesar,DPS Sector 103,DPSG Palam Vihar,DPSG Palam Vihar Gurugram,DSD College,Damdama Lake,Damdama Lake Rd,Damdama More,Daultabad Stadium,Daultabad Village Park,De Adventure Amusement Park,De Adve

In [118]:
cosine_sim3 = cosine_similarity(location_df_normalized)

In [119]:
cosine_sim3.shape

(246, 246)

In [120]:
# analysing recommendation of three recommender by adding weight to each 

In [121]:
df.head()

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...",
1,M3M Crown,"3, 4 BHK Apartment in Sector 111, Gurgaon","['DPSG Palam Vihar Gurugram', 'The NorthCap Un...","{'DPSG Palam Vihar Gurugram': '1.4 Km', 'The N...",https://www.99acres.com/m3m-crown-sector-111-g...,"{'3 BHK': {'building_type': 'Apartment', 'area...",
2,Adani Brahma Samsara Vilasa,"Land, 3, 4 BHK Independent Floor in Sector 63,...","['AIPL Business Club Sector 62', 'Heritage Xpe...","{'AIPL Business Club Sector 62': '2.7 Km', 'He...",https://www.99acres.com/adani-brahma-samsara-v...,{'3 BHK': {'building_type': 'Independent Floor...,
3,Sobha City,"2, 3, 4 BHK Apartment in Sector 108, Gurgaon","['The Shikshiyan School', 'WTC Plaza', 'Luxus ...","{'The Shikshiyan School': '2.9 KM', 'WTC Plaza...",https://www.99acres.com/sobha-city-sector-108-...,"{'2 BHK': {'building_type': 'Apartment', 'area...",
4,Signature Global City 93,"2, 3 BHK Independent Floor in Sector 93 Gurgaon","['Pranavananda Int. School', 'DLF Site central...","{'Pranavananda Int. School': '450 m', 'DLF Sit...",https://www.99acres.com/signature-global-city-...,{'2 BHK': {'building_type': 'Independent Floor...,


In [122]:
location_df_normalized.head()

,AIIMS,AIIMS Jhajjar,AIPL Business Centre,AIPL Business Club,AIPL Business Club Sector 62,AIPL Business Co Working Space,AIPL Business Tower,AIPL Joy Street Mall,APJ Abdul Kalam Park,ASF Insignia SEZ,Aapno Ghar,AapnoGhar,Aarvy Healthcare,Aarvy Healthcare Hospital,Aarvy Healthcare Super Speciality,Aarvy Hospital,Aatish Hospital,Accenture DDC5,Adarsh Senior Secondary School,"Adarsh public school,Garhi Harsaru",Agri Business Management Collage,Airia Mall,Airia Mall Sector 68,Airport,Ajit Stadium Dhanwapur,Alfaa Health Care Hospital,Alpine Convent School,Alpine Hospital,Alpine School,Altrade Business Centre,Aman Hospital,Aman Hospital & Surgical Centre,Ambience Mall,Ambience Mall New,Ambience Public School,American Express,Amity,Amity University,Amity University Gurugram,Amma Hospital,Anand Multispeciality Hospital,Anand Preschool,Ananta Hospital,Ansal Plaza,Anya Gurgaon,Ap Sports cricket ground,Apex Plus Hospital,Apollo,Apollo Pharmacy,Approved Sector 37 Mero Station,Appu Ghar,Appu Ghar Water Park,Aradhya Cricket Club Gurgaon,Aravali Adventure Hill,Aravalli Hill View Point,Aravalli Hills,Arc Hospital,Arc Multi Speciality,Arc Multi Speciality Hospital,Ardee Mall,Artemis,Artemis Hospital,Artemis Hospital Gurgaon,Artimis hospital,Aryan Hospital,Ascendas OneHub Gurgaon,Ascendas OneHub Gurgaon Business Park,Ashiana Anmol Kid Centric Homes,Ashoka International School,Athena,Au Grand Air,Axis Bank,Axis Bank ATM,"Axis Bank, Sohna Rd",BM College of Technology,BM College of Technology & Mgmt,BML Munjal University (BMU),BOB ATM,BSF Golf Course,Baba Kanala Chowk,Badsa AMS Hospital,Badshahpur Sohna Hwy,Badshahpur Sohna Rd Hwy,"Badshahpur Sohna Rd Hwy, Haryana","Badshahpur Sohna Rd Hwy, Malibu Town","Badshahpur Sohna Rd Hwy, Raghav Vatika","Badshahpur Sohna Rd Hwy, Rajoria Ngr","Badshahpur Sohna Rd Hwy,Sector 48",Baghera University,Bajghera Road,Bal Bharati Public School,Balaji Hospital,Bamroli Cricket Ground,Banjara Market Gurugram,Bank Of Baroda,Bank Of Baroda ATM,Basai Dhancourt Railway Station,Basai Dhankot,Basai Dhankot Railway Station,Basai Enclave Park,Basai Metro Station,Basai Road,Basant Valley Global School,Best IVF Centre,Bestech Business Tower,Bestech Central Square Mall,Bharat Petrol Pump,Bharat Petroleum Petrol Pump,Bharat Petroleum Retail Outlet,Bharat Petroleum Shree Shyam Filling,Bharat Ram Global School,Bharat Singh fuel company,Bharti International Convent School,Bhondsi Nature Park,BigBazaar,Bijwasan Railway Station,Biryani Shah,Blue Bells Public School,Broadways International School,CBR Cricket Ground,CD International School,CK Birla Hospital,"CK Birla Hospital, Gurgaon",CNG Petrol Pump,Cambridge College Of Education,Cambridge Montessori,Cambridge Montessori Preschool,Cambridge Pre-School,Canara Bank,Canara Bank - Nawada Fatehpur,Canara Bank ATM,Canara Bank New Palam Vihar,"Candor TechSpace, Sector 48",Candor Techspace,Capital Business Park,Capital Cyberscape,Captain Chandan Lal Marg,Central Bank Of India Sohna Rd,Central Park Flower Valley,Central Park II Road,Central Park Resorts,"Central Park, Sohna Rd",Central Peripheral Road,Central Plaza Mall,Centrum Plaza,Chauma Road,Cherub's Cradle,Children Park,Chirag Hospital,Choice pharmacy,Citibank ATM,City Hospital,City Square,Civil Hospital,Cloudnine Hospital Sector 47,CoNexus.Life B35,Colonel's Central Academy,Conscient One,Conscient One Mall,Cool Deck Coffee,Country Inn,Country Inn & Suites by Radisson,Creative Tennis Academy,Cricket Academy,Cyber ​​Park,DLC Cricket Ground,DLF Corporate Greens,DLF Corporate Park,DLF Cyber City,DLF Golf and Country Club,DLF Grand Mall,DLF Linear Park,DLF Site central office,DLF World Tech Park,DLF5 Summit Plaza,DPG Degree College,DPG Institute of Technology,DPGITM Engineering College Sector 34,DPS,DPS International Edge,DPS International School,DPS Manesar,DPS Sector 103,DPSG Palam Vihar,DPSG Palam Vihar Gurugram,DSD College,Damdama Lake,Damdama Lake Rd,Damdama More,Daultabad Stadium,Daultabad Village Park,De Adventure Amusement Park,De Adve

In [149]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    cosine_sim_matrix = 30*cosine_sim1 + 20*cosine_sim2 + 8*cosine_sim3
    # cosine_sim_matrix = cosine_sim3
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('M3M Crown').head(10)

,PropertyName,SimilarityScore
0,DLF The Ultima,29.407125
1,Bestech Altura,24.879864
2,Emaar Gurgaon Greens,23.476696
3,M3M Merlin,21.817000
4,Tulip Violet,21.187416
5,Bestech Park View Sanskruti,20.689666
6,DLF The Primus,20.473484
7,Saan Verdante,20.428134
8,Conscient Heritage Max,19.983139
9,Silverglades Hightown Residences,18.927742


In [125]:
# now i am adding also one more recommender of sector and test on same property and checking performance

In [126]:
df.head(1)

,PropertyName,PropertySubName,NearbyLocations,LocationAdvantages,Link,PriceDetails,TopFacilities
0,Smartworld One DXP,"2, 3, 4 BHK Apartment in Sector 113, Gurgaon","['Bajghera Road', 'Palam Vihar Halt', 'DPSG Pa...","{'Bajghera Road': '800 Meter', 'Palam Vihar Ha...",https://www.99acres.com/smartworld-one-dxp-sec...,"{'2 BHK': {'building_type': 'Apartment', 'area...",


In [136]:
import re

def extract_sector(text):
    if pd.isna(text):
        return None

    match = re.search(r'(Sector\s*-?\s*\d+[A-Za-z]?)', str(text), flags=re.IGNORECASE)

    if match:
        sector = match.group(1)
        # Standardize spacing
        sector = re.sub(r'\s+', ' ', sector)
        sector = re.sub(r'\s*-\s*', ' ', sector)
        return sector.title()

    return None

In [137]:
df['sector']=df['PropertySubName'].apply(extract_sector)

In [144]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fill missing values
sector_df = df[['PropertyName', 'sector']].copy()
sector_df['sector'] = sector_df['sector'].fillna('Unknown')

# TF-IDF
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(sector_df['sector'])

# Cosine Similarity
cosine_sim4 = cosine_similarity(tfidf_matrix)

In [147]:
cosine_sim4

array([[1.        , 0.04297783, 0.04724741, ..., 0.04528469, 0.05455475,
        0.04724741],
       [0.04297783, 1.        , 0.04484056, ..., 0.04297783, 0.05177565,
        0.04484056],
       [0.04724741, 0.04484056, 1.        , ..., 0.04724741, 0.05691924,
        0.04929519],
       ...,
       [0.04528469, 0.04297783, 0.04724741, ..., 1.        , 0.05455475,
        0.04724741],
       [0.05455475, 0.05177565, 0.05691924, ..., 0.05455475, 1.        ,
        0.05691924],
       [0.04724741, 0.04484056, 0.04929519, ..., 0.04724741, 0.05691924,
        1.        ]], shape=(246, 246))

In [146]:
def recommend_sector(property_name):
    
    # Get similarity scores
    sim_scores = list(enumerate(
        cosine_sim4[sector_df.index[sector_df['PropertyName'] == property_name][0]]
    ))
    
    # Sort by similarity
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Top 5 recommendations (excluding itself)
    top_properties = sorted_scores[1:6]
    
    recommendations_df = pd.DataFrame({
        'PropertyName': [sector_df.iloc[i[0]]['property_name'] for i in top_properties],
        'Sector': [sector_df.iloc[i[0]]['sector'] for i in top_properties],
        'SimilarityScore': [i[1] for i in top_properties]
    })
    
    return recommendations_df

In [153]:
def recommend_properties_with_scores(property_name, top_n=247):
    
    cosine_sim_matrix = 30*cosine_sim1 + 20*cosine_sim2 + 8*cosine_sim3 + 50*cosine_sim4
    # cosine_sim_matrix = cosine_sim3
    
    # Get the similarity scores for the property using its name as the index
    sim_scores = list(enumerate(cosine_sim_matrix[location_df_normalized.index.get_loc(property_name)]))
    
    # Sort properties based on the similarity scores
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the indices and scores of the top_n most similar properties
    top_indices = [i[0] for i in sorted_scores[1:top_n+1]]
    top_scores = [i[1] for i in sorted_scores[1:top_n+1]]
    
    # Retrieve the names of the top properties using the indices
    top_properties = location_df_normalized.index[top_indices].tolist()
    
    # Create a dataframe with the results
    recommendations_df = pd.DataFrame({
        'PropertyName': top_properties,
        'SimilarityScore': top_scores
    })
    return recommendations_df

# Test the recommender function using a property name
recommend_properties_with_scores('M3M Crown').head(10)

,PropertyName,SimilarityScore
0,Puri Diplomatic Greens,52.656088
1,DLF The Ultima,31.731292
2,Bestech Altura,27.346218
3,Emaar Gurgaon Greens,25.875011
4,M3M Merlin,24.141166
5,Tulip Violet,23.336307
6,Bestech Park View Sanskruti,22.838558
7,Saan Verdante,22.577025
8,DLF The Primus,22.512908
9,Conscient Heritage Max,22.381454


In [ ]:
# PropertyName	SimilarityScore
# 0	DLF The Ultima	29.407125
# 1	Bestech Altura	24.879864
# 2	Emaar Gurgaon Greens	23.476696
# 3	M3M Merlin	21.817000
# 4	Tulip Violet	21.187416
# 5	Bestech Park View Sanskruti	20.689666
# 6	DLF The Primus	20.473484
# 7	Saan Verdante	20.428134
# 8	Conscient Heritage Max	19.983139
# 9	Silverglades Hightown Residences	18.927742
